# 🚀 Automatización Masiva de Imágenes con FLUX.1 en Google Colab Pro (GPU A100)
### Guía de Alta Velocidad (2 a 3 segundos por imagen, Formato WebP 16:9 SEO-Ready)

Este cuaderno está configurado para la **GPU NVIDIA A100 (40GB / 80GB)** de Google Colab Pro. Permite generar las 537 imágenes editoriales de Todo Lima en **solo ~25 a 30 minutos** (frente a las 12 horas que tardaba en la GPU T4 básica).

---
### 📋 Arquitectura de Trabajo (Solo 2 Archivos)
Solo necesitas tener en tu Google Drive dentro de la carpeta `Todo_Lima_Flux`:
1. **`prompts.txt`**: Un prompt fotográfico refinado por línea (sin 16:9, sin textos falsos, sin trampas de manos).
2. **`nombres.txt`**: El nombre del archivo limpio por línea (sin extensiones).

---
### ⚡ Ventajas Técnicas en GPU A100:
* **Velocidad Extrema:** ~2.5 segundos por imagen en 4 pasos nativos.
* **Zero Cuello de Botella:** FLUX cabe completo en la memoria HBM2e sin transferencias lentas.
* **Manos y Rostros Limpios:** `guidance_scale=0.0` para evitar saturación y dedos en garra.
* **Reanudación Automática:** Si borras alguna imagen que no te gustó, al darle Play solo regenera las que falten.

---
## ⚙️ Paso 1: Montaje de Drive e Instalación Rápida
Ejecuta esta celda una sola vez al iniciar la sesión con GPU A100 activa.

In [ ]:
# =====================================================================
# PASO 1: MONTAJE DE DRIVE E INSTALACIÓN DE LIBRERÍAS COMPATIBLES
# =====================================================================
from google.colab import drive
drive.mount('/content/drive')

# Instalación de librerías con aceleración oficial
!pip install -q "huggingface-hub<2.0,>=0.25.0" "diffusers>=0.30.0" transformers accelerate sentencepiece protobuf pillow
print('✓ Entorno listo y librerías optimizadas para GPU A100.')

---
## 🎨 Paso 2: Generación Masiva Ultrarrápida (A100 + WebP + Guidance 0.0)
Esta celda procesa el lote completo a toda velocidad. Si detienes la ejecución o borras imágenes defectuosas en tu Drive, al volver a ejecutar **solo regenerará las que no existan**.

In [ ]:
# =====================================================================
# PASO 2: GENERADOR MASIVO EN GPU A100 (2-3 SEGUNDOS POR IMAGEN)
# =====================================================================
import os
import torch
from diffusers import FluxPipeline
from PIL import Image

# Optimización de memoria en CUDA
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# ==========================================
# RUTAS EN GOOGLE DRIVE
# ==========================================
CARPETA_PROYECTO = '/content/drive/MyDrive/Todo_Lima_Flux'
MODELO_LOCAL = os.path.join(CARPETA_PROYECTO, 'FLUX.1-schnell')
OUTPUT_DIR = os.path.join(CARPETA_PROYECTO, 'imagenes_generadas')
os.makedirs(OUTPUT_DIR, exist_ok=True)

PROMPTS_FILE = os.path.join(CARPETA_PROYECTO, 'prompts.txt')
NAMES_FILE = os.path.join(CARPETA_PROYECTO, 'nombres.txt')

# Validación de archivos de entrada
for ruta, desc in [(PROMPTS_FILE, 'prompts.txt'), (NAMES_FILE, 'nombres.txt')]:
    if not os.path.exists(ruta):
        raise FileNotFoundError(f'No se encontró el archivo {desc} en: {ruta}')

with open(PROMPTS_FILE, 'r', encoding='utf-8') as f:
    prompts = [line.strip() for line in f if line.strip()]

with open(NAMES_FILE, 'r', encoding='utf-8') as f:
    nombres = [line.strip() for line in f if line.strip()]

total = len(prompts)
assert len(prompts) == len(nombres), f'Desalineación: Hay {len(prompts)} prompts y {len(nombres)} nombres.'
print(f'✓ Verificado: {total} imágenes en la lista de procesamiento.')

# ==========================================
# CARGA DE FLUX.1-schnell (MÁXIMA CALIDAD BFLOAT16 EN A100)
# ==========================================
print('Cargando FLUX.1-schnell a la memoria de la GPU A100...')
pipe = FluxPipeline.from_pretrained(
    MODELO_LOCAL,
    torch_dtype=torch.bfloat16
)

# En GPU A100, model_cpu_offload vuela a máxima velocidad
pipe.enable_model_cpu_offload()
pipe.vae.enable_slicing()
pipe.vae.enable_tiling()
print('✓ Pipeline listo a máxima velocidad.')

# ==========================================
# BUCLE DE GENERACIÓN MASIVA CONTINUA
# ==========================================
for i, (prompt, nombre_raw) in enumerate(zip(prompts, nombres)):
    nombre_limpio = nombre_raw.replace('.webp', '').replace('.png', '').replace('.jpg', '')
    nombre_seguro = ''.join(c for c in nombre_limpio if c.isalnum() or c in (' ', '_', '-')).strip()
    
    archivo_final = f'{nombre_seguro}.webp'
    ruta_guardado = os.path.join(OUTPUT_DIR, archivo_final)
    
    # Si ya existe en Drive, salta para no gastar tiempo
    if os.path.exists(ruta_guardado):
        print(f'[{i+1}/{total}] Ya existe, saltando: {archivo_final}')
        continue

    print(f'[{i+1}/{total}] Generando (A100 ~2.5s): {archivo_final}...')
    
    # Inferencia en formato 16:9 (1024x576) con guidance 0.0 nativo para evitar deformaciones
    image = pipe(
        prompt=prompt,
        num_inference_steps=4,
        guidance_scale=0.0,
        width=1024,
        height=576
    ).images[0]
    
    # Guardado nativo WebP calidad 85 directo para SEO web
    image.save(ruta_guardado, format='WEBP', quality=85)

print('\n🎉 ¡Lote completo generado con éxito en tu Google Drive!')

---
## 💡 Preguntas Frecuentes y Trucos de Producción

### ¿Cómo regenerar solo una imagen específica si no me gusta?
1. Abre la carpeta `imagenes_generadas` en tu Google Drive.
2. Elimina la imagen que quieras rehacer.
3. Dale play a la Celda 2 en Colab. El script revisará todas las que ya existen, las saltará y **únicamente generará la que eliminaste**.

### ¿Cómo rehacer todas las 537 imágenes desde cero con los prompts refinados?
Simplemente borra todo el contenido de la carpeta `imagenes_generadas` en Google Drive y dale play a la Celda 2. En la GPU A100, todo el lote de 537 imágenes estará listo en **aproximadamente 25 minutos**.